<a href="https://colab.research.google.com/github/sinaganjoie82-cmyk/California-Crash-Severity-Spatial-Analysis/blob/main/scripts/Crash_Analysis_Model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# 1. Methodology Setup (Loading Libraries)
library(tidyverse)
library(sf)
library(sp)
library(GWmodel)
library(leaflet)
library(ranger)
library(viridis)
library(ggplot2)

# 2. Data Loading & Feature Engineering
# Load Data (Check if file exists)
if(file.exists("US_Accidents_March23.csv")){
  raw_data <- read_csv("US_Accidents_March23.csv", show_col_types = FALSE)
} else {
  stop("File not found! Please check US_Accidents_March23.csv")
}

# Rename Columns if needed (Handling different naming conventions)
if ("Temperature (F)" %in% names (raw_data)) {
  colnames (raw_data) [which (names (raw_data) == "Temperature (F)")] <- "Temperature"
}
if("Humidity(%)" %in% names(raw_data)) {
  colnames (raw_data) [which(names (raw_data) == "Humidity(%)")] <- "Humidity"
}
if("Visibility(mi)" %in% names (raw_data)) {
  colnames (raw_data) [which(names(raw_data) == "Visibility(mi)")] <- "Visibility"
}

# Filter & Feature Engineering for California
df_ca <- raw_data %>%
  filter(State == "CA") %>%
  filter(!is.na(Start_Lat) & !is.na(Start_Lng)) %>%
  filter(!is.na(Severity) & !is.na(Temperature)) %>%
  select(ID, Severity, Start_Lat, Start_Lng, Start_Time, Temperature, Visibility, Humidity, Weather_Condition, Junction, Crossing) %>%
  mutate(
    Start_Time = as.POSIXct(Start_Time, format="%Y-%m-%d %H:%M:%S"),
    Hour = as.numeric(format(Start_Time, "%H")),
    Is_Night = ifelse(Hour >= 19 | Hour < 6, 1, 0),
    Is_Severe = ifelse(Severity >= 3, 1, 0),
    Is_Severe_Factor = as.factor(Is_Severe)
  ) %>%
  na.omit()

# Clear memory
rm(raw_data)
gc()
print(paste("Total Cleaned Records:", nrow(df_ca)))

# 3. Interactive Map
set.seed(123)
map_sample <- df_ca %>% sample_n(1000)
pal <- colorFactor(c("navy", "red"), domain = c(0, 1))

leaflet(map_sample) %>%
  addTiles() %>%
  addCircleMarkers(
    ~Start_Lng, ~Start_Lat,
    color = ~pal(Is_Severe),
    radius = 4, stroke = FALSE, fillOpacity = 0.8,
    popup = ~paste("Severity:", Severity, "<br>Temp:", Temperature)
  ) %>%
  addLegend("bottomright", pal = pal, values = ~Is_Severe,
            title = "Severity", labels = c("Minor", "Severe"))

# 4. Machine Learning (Random Forest)
set.seed(42)
ml_data <- df_ca %>% sample_n(min(20000, nrow(df_ca)))
rf_model <- ranger(Is_Severe_Factor ~ Temperature + Visibility + Humidity + Hour + Is_Night + Junction + Crossing,
                   data = ml_data,
                   importance = 'impurity',
                   num.trees = 100)

imp_df <- data.frame(Variable = names(rf_model$variable.importance),
                     Importance = rf_model$variable.importance)

ggplot(imp_df, aes(x = reorder(Variable, Importance), y = Importance)) +
  geom_bar(stat = "identity", fill = "#2c3e50") +
  coord_flip() +
  theme_minimal() +
  labs(title = "Feature Importance", x = "Variables", y = "Score")

# 5. Spatial Heterogeneity (GWR)
set.seed(99)
gwr_data <- df_ca %>% sample_n(3000)
sp_data <- gwr_data
coordinates(sp_data) <- ~Start_Lng+Start_Lat

# GWR Model
bw <- bw.gwr(Is_Severe ~ Temperature, data = sp_data, approach = "AIC",
             kernel = "gaussian", adaptive = TRUE)

gwr_model <- gwr.basic(Is_Severe ~ Temperature, data = sp_data,
                       bw = bw, kernel = "gaussian", adaptive = TRUE)

# Plot Results
gwr_results <- as.data.frame(gwr_model$SDF)

ggplot() +
  geom_point(data = gwr_results,
             aes(x = Start_Lng, y = Start_Lat, color = Temperature),
             size = 1.5, alpha = 0.9) +
  scale_color_viridis(option = "magma", name = "Temp Coeff") +
  theme_void() +
  labs(title = "Spatial Heterogeneity: Temperature Impact",
       subtitle = "Yellow areas = Stronger impact") +
  coord_fixed()